# 05 — Protocol Version Propagation and Dependency Staleness Check (from scratch, offline)

Companion notebook to `../06-protocol-amendment-versioning-and-document-revision-handling.md`.

This notebook implements, in plain Python, the proposed design from Chapter 06:

1. Every generated document (ICF section, PLPS section, SOC entry) carries the `protocol_version`
   it was generated from/about.
2. A `flag_stale_dependents` check marks any ICF/PLPS document whose `protocol_version` trails the
   Protocol's current version as `needs_regeneration` — making staleness a visible, tracked state
   instead of a silent gap.
3. An enforced invariant — a SOC entry's `diffs_against_version` must equal `protocol_version - 1` —
   guarantees the SOC chain never skips a version, even when amendments land in quick succession.

Extends the same "mock generation pipeline" pattern as `04_protocol_comparison_pipeline.ipynb`, but
for the version/amendment axis instead of the single-generation-and-grounding-check axis. Fully
offline: standard library only, no model calls, no API keys.

**Reminder consistent with the rest of this course**: this is an illustrative reconstruction of a
plausible design, not a description of a verified production system — there is no source repository
behind this course. The mechanics below are built to be technically precise and directly runnable,
which is what makes them useful to reason about in an interview, not a claim that this exact code
exists in a real codebase.

## 1. Modeling generated documents with a `protocol_version` field

A tiny in-memory "database" of generated documents. Each one records which module it belongs to,
which Protocol section it covers, which `protocol_version` it was generated from, and its current
`review_status`. SOC entries additionally record `diffs_against_version` — the prior version they
diff against.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class GeneratedDocument:
    doc_id: str
    module: str          # "ICF" | "PLPS" | "SOC"
    section_id: str
    protocol_version: int
    review_status: str = "pending_review"   # "pending_review" | "approved" | "needs_regeneration"
    diffs_against_version: Optional[int] = None   # SOC entries only


# Documents generated when the Protocol was at version 1 (first draft, nothing amended yet)
documents: list[GeneratedDocument] = [
    GeneratedDocument("ICF-3.1-v1", "ICF", "3.1", protocol_version=1, review_status="approved"),
    GeneratedDocument("PLPS-5.1-v1", "PLPS", "5.1", protocol_version=1, review_status="approved"),
]

for d in documents:
    print(d)


## 2. Simulating a protocol amendment: version 1 -> version 2

Section 5.1 (dosing) is amended. A realistic pipeline (Chapter 06, Part 1) re-parses the amended
Protocol, regenerates the ICF/PLPS sections the amendment touches, and generates a SOC entry
diffing the new section 5.1 text against the immediately-prior version. Section 3.1 (eligibility)
was untouched by this particular amendment, so nothing regenerates it yet -- which is exactly the
scenario `flag_stale_dependents` (below) needs to reason about correctly: is 3.1 stale, or simply
unaffected?

In [ ]:
CURRENT_PROTOCOL_VERSION = {"demo-protocol": 1}


def amend_protocol(protocol_id: str, amended_section_ids: set[str], documents: list[GeneratedDocument]):
    """Bump the protocol's version and regenerate ONLY the documents covering an amended
    section -- mirroring the realistic 'narrow the regeneration, don't blindly redo everything'
    approach from Chapter 06, Part 1."""
    new_version = CURRENT_PROTOCOL_VERSION[protocol_id] + 1
    CURRENT_PROTOCOL_VERSION[protocol_id] = new_version

    for doc in documents:
        if doc.module in ("ICF", "PLPS") and doc.section_id in amended_section_ids:
            # a realistic pipeline would regenerate content here; we only need the
            # bookkeeping (protocol_version bump + status reset) for this demo
            doc.protocol_version = new_version
            doc.review_status = "pending_review"

    # one SOC entry per amended section, diffing against the immediately-prior version
    soc_entries = [
        GeneratedDocument(
            doc_id=f"SOC-{sid}-v{new_version}",
            module="SOC",
            section_id=sid,
            protocol_version=new_version,
            diffs_against_version=new_version - 1,
            review_status="pending_review",
        )
        for sid in sorted(amended_section_ids)
    ]
    documents.extend(soc_entries)
    return new_version, soc_entries


new_version, soc_entries = amend_protocol("demo-protocol", {"5.1"}, documents)
print(f"Protocol bumped to version {new_version}")
for d in documents:
    print(" ", d)

assert CURRENT_PROTOCOL_VERSION["demo-protocol"] == 2
assert any(d.doc_id == "SOC-5.1-v2" and d.diffs_against_version == 1 for d in soc_entries)
print("\nOK: 5.1's ICF/PLPS bumped to v2, a SOC-5.1-v2 entry was generated diffing against v1, "
      "and 3.1 (untouched by this amendment) was left exactly as it was.")


## 3. `flag_stale_dependents`: making staleness a visible, tracked state

Directly from Chapter 06, Part 4. This is the mechanism that answers "how do you make sure nothing
silently still reflects the old version": any ICF/PLPS document whose `protocol_version` trails the
Protocol's current version is flagged `needs_regeneration` -- not silently left as-is, and not
conflated with "approved" just because it was approved *at some point*.

In [ ]:
def flag_stale_dependents(current_protocol_version: int, documents: list[GeneratedDocument]):
    """Any ICF/PLPS document whose protocol_version trails the Protocol's current version is
    stale, and staleness is a distinct, first-class status -- not silently indistinguishable
    from 'approved' or 'pending_review'."""
    flagged = []
    for doc in documents:
        if doc.module in ("ICF", "PLPS") and doc.protocol_version < current_protocol_version:
            doc.review_status = "needs_regeneration"
            flagged.append(doc.doc_id)
    return flagged


flagged = flag_stale_dependents(CURRENT_PROTOCOL_VERSION["demo-protocol"], documents)
print("Flagged as needs_regeneration:", flagged)

# ICF-3.1-v1 is still at v1 while the protocol is now at v2 -- it must be flagged
assert "ICF-3.1-v1" in flagged
# PLPS-5.1 and ICF-5.1 were already regenerated to v2 by amend_protocol() above -- NOT stale
assert not any(d.doc_id.startswith("PLPS-5.1") and d.doc_id in flagged for d in documents)
print("\nOK: the untouched 3.1 documents are correctly flagged stale relative to the new protocol "
      "version, while the 5.1 documents that were actually regenerated are not -- this is exactly "
      "the distinction between 'the SOC accurately lists what changed' and 'every dependent has "
      "actually been regenerated' from Chapter 06, Part 3: two different checks, over two different "
      "sets of documents, that can each fail independently.")


## 4. Clearing the flag: regeneration is the only thing that resolves staleness

A `needs_regeneration` flag is cleared only by an actual regeneration event -- never by a status
edit alone. This keeps the flag meaningful: if it's clear, a document really was rebuilt against
the current protocol version, not just marked as such.

In [ ]:
def regenerate(doc: GeneratedDocument, new_protocol_version: int):
    """Stand-in for an actual regeneration call through the generation pipeline
    (Chapter 05's per-section-type templates + grounding check) -- what matters for this
    notebook's purposes is that protocol_version is updated and status resets to
    pending_review, i.e. it re-enters the SAME human-review gate a first draft would."""
    doc.protocol_version = new_protocol_version
    doc.review_status = "pending_review"


stale_doc = next(d for d in documents if d.doc_id == "ICF-3.1-v1")
regenerate(stale_doc, CURRENT_PROTOCOL_VERSION["demo-protocol"])
print("After regeneration:", stale_doc)

assert stale_doc.protocol_version == 2
assert stale_doc.review_status == "pending_review"   # NOT auto-approved -- still needs a human reviewer
print("\nOK: regenerating a stale document brings its protocol_version current and resets it to "
      "pending_review -- it does NOT jump straight to 'approved'. An amendment-triggered "
      "regeneration goes through the identical review gate a first-time generation would "
      "(Chapter 06, Part 5) -- a version bump is never a fast path around human review.")


## 5. Rapid amendments: the SOC chain must never skip a version

The detail Chapter 06 calls out as the one a sloppy implementation gets wrong: if a second
amendment lands before the first amendment's regeneration cycle has even finished, the SOC for the
new version must diff against the *immediately-prior* version -- never skip back further, even
though skipping would technically still produce *a* diff. The invariant below is enforced in code,
not just assumed.

In [ ]:
def make_soc_entry(section_id: str, protocol_version: int, diffs_against_version: int):
    """Construct a SOC entry, enforcing the invariant from Chapter 06, Part 4: a SOC entry
    must diff against the version immediately prior to the one it's generated at. Raises
    instead of silently accepting a version-skipping SOC -- exactly the kind of bug that could
    otherwise bury an intermediate, safety-relevant amendment inside a bigger, noisier diff."""
    if diffs_against_version != protocol_version - 1:
        raise ValueError(
            f"SOC entry for v{protocol_version} must diff against v{protocol_version - 1}, "
            f"not v{diffs_against_version} -- this would skip a version and could bury an "
            f"intermediate amendment's changes inside a larger, noisier diff."
        )
    return GeneratedDocument(
        doc_id=f"SOC-{section_id}-v{protocol_version}",
        module="SOC",
        section_id=section_id,
        protocol_version=protocol_version,
        diffs_against_version=diffs_against_version,
        review_status="pending_review",
    )


# A second amendment lands quickly, bumping the protocol straight from v2 to v3
new_version_2, _ = amend_protocol("demo-protocol", {"3.1"}, documents)
assert new_version_2 == 3
print(f"Protocol bumped to version {new_version_2} (a second amendment, shortly after the first)")

# CORRECT: the SOC for v3 diffs against v2 (immediately prior) -- this must succeed
correct_soc = make_soc_entry("3.1", protocol_version=3, diffs_against_version=2)
print("Correctly-chained SOC entry:", correct_soc)

# BUGGY: an implementation that (incorrectly) diffs v3 straight against v1, skipping v2 entirely
try:
    make_soc_entry("3.1", protocol_version=3, diffs_against_version=1)
    raised = False
except ValueError as e:
    raised = True
    print("\nRejected a version-skipping SOC entry, as expected:")
    print(" ", e)

assert raised, "a version-skipping SOC entry (v3 vs v1, skipping v2) must be rejected, not silently accepted"
print("\nOK: the immediately-prior-version invariant is enforced in code -- a SOC entry that would "
      "skip v2 and diff v3 straight against v1 is rejected outright, exactly the protection Chapter "
      "06 calls for when amendments happen in quick succession.")


## 6. Tying it back

- `protocol_version` propagated onto every generated document (Section 1-2) is what makes staleness
  a *checkable* question at all -- without it, "is this ICF current" has no defined answer.
- `flag_stale_dependents` (Section 3) is the concrete mechanism behind "make sure nothing silently
  still reflects the old version" -- it turns an invisible gap into a visible, reviewer-facing queue
  item, and Section 3's assertions show it correctly distinguishes documents that were actually
  regenerated from documents that were merely left alone.
- The `make_soc_entry` invariant (Section 5) is what prevents rapid, back-to-back amendments from
  silently collapsing into a single, version-skipping SOC diff that could bury an intermediate,
  safety-relevant change.
- Regeneration always resets a document to `pending_review` (Section 4), never straight to
  `approved` -- consistent with Chapter 06, Part 5: an amendment-triggered regeneration is never a
  fast path around the same human regulatory-affairs review gate a first-time generation goes
  through.